# HCLA RQ2 — ONE-CLICK Evidence Groundedness

이 노트북은 **RQ1을 다시 학습하지 않고**, 기존 `rq1_multimodel_results_3seeds.pkl`을 자동으로 찾아 RQ2만 실행합니다.

### RQ2가 검증하는 것
- 동일한 100개 사례(정상 50 + 이상 50)에 대해 **Baseline 설명과 HCLA 설명**을 각각 3회 생성
- 각 설명을 atomic claim으로 분해
- Llama Judge가 각 claim을 `SUPPORTED / UNSUPPORTED / CONTRADICTED`로 판정
- 최종적으로 **ESR / UCR / CR / EC**를 비교해 HCLA 설명의 evidence-groundedness를 평가

### 실행 방법
**Colab에서 `런타임 → 모두 실행`만 누르면 됩니다.**
- RQ1 PKL: 기존 기본 경로 우선 → 없으면 Drive에서 자동 탐색
- 원본 CSV: 기존 기본 경로 우선 → 없으면 Drive에서 자동 탐색
- OpenRouter Key: 환경변수 → Colab Secret → 없으면 입력창 순서로 자동 처리
- 중간 중단 시 checkpoint를 읽고 이어서 실행

**본 실험 규모: 100 cases × 2 conditions × 3 generations = 600 explanations**


In [ ]:
# Colab 최초 1회
!pip -q install -U numpy pandas scikit-learn matplotlib xgboost lightgbm shap torch openai

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, json, re, time, random, pickle, tempfile, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import shap
from openai import OpenAI

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 1. RQ1 PKL 호환용 클래스 정의
RQ1 PKL에 TabularResNet/FT-Transformer 객체가 포함되어 있어도 `pickle.load()`가 가능하도록 원본 클래스명을 그대로 정의합니다.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d, d), nn.BatchNorm1d(d),
        )
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(x + self.net(x))

class TabularResNetNet(nn.Module):
    def __init__(self, n_features, d=128, n_blocks=3, dropout=0.1):
        super().__init__()
        self.input = nn.Sequential(nn.Linear(n_features, d), nn.BatchNorm1d(d), nn.ReLU())
        self.blocks = nn.Sequential(*[ResidualBlock(d, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(d, 1)
    def forward(self, x):
        return self.head(self.blocks(self.input(x))).squeeze(1)

class NumericTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class FTTransformerNet(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.tokenizer = NumericTokenizer(n_features, d_token)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, 1)
    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls.expand(x.shape[0], -1, -1)
        z = self.encoder(torch.cat([cls, tok], dim=1))
        return self.head(self.norm(z[:, 0])).squeeze(1)

class TorchBinaryClassifier:
    def __init__(self, architecture, seed=42, epochs=20, batch_size=4096, patience=5):
        self.architecture = architecture
        self.seed = seed
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.scaler = StandardScaler()
        self.model = None
    def predict_proba(self, X):
        Xs = self.scaler.transform(X).astype(np.float32)
        loader = DataLoader(TensorDataset(torch.tensor(Xs, dtype=torch.float32)), batch_size=self.batch_size, shuffle=False)
        self.model = self.model.to(DEVICE)
        self.model.eval()
        probs = []
        with torch.no_grad():
            for (xb,) in loader:
                p = torch.sigmoid(self.model(xb.to(DEVICE))).cpu().numpy()
                probs.append(p)
        p1 = np.concatenate(probs)
        return np.column_stack([1 - p1, p1])

## 2. 자동 경로 / 본 실험 설정
코드 수정 없이 기존 프로젝트 경로를 우선 사용하고, 없으면 Google Drive에서 자동 탐색합니다.


In [ ]:

import os, glob

# ------------------------------------------------------------
# 본 실험 설정: 수정 불필요
# ------------------------------------------------------------
RQ2_EXPERIMENT = "E2_Random_AddressRemoved"
RQ2_MODEL_SEED = 42
RQ2_SPLIT_SEED = 42

N_NORMAL = 50
N_ANOMALY = 50
TOP_K_EVIDENCE = 5
N_GENERATIONS = 3
RQ2_PILOT = False

SHAP_BACKGROUND_N = 200
SHAP_GLOBAL_N = 1000

# ------------------------------------------------------------
# 1) 기존 프로젝트의 표준 경로를 먼저 확인
# ------------------------------------------------------------
DEFAULT_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_OpenRouter_SHAP"
DEFAULT_PKL = os.path.join(DEFAULT_OUTPUT_DIR, "rq1_multimodel_results_3seeds.pkl")
DEFAULT_DATA = "/content/drive/MyDrive/Colab Notebooks/Data/BTC/ETH MixingTornadoCash DF.csv"

def find_one(preferred, patterns, label):
    if os.path.exists(preferred):
        print(f"✅ {label} 기본 경로 사용")
        return preferred

    matches = []
    for pat in patterns:
        matches.extend(glob.glob(pat, recursive=True))
    matches = sorted(set(p for p in matches if os.path.isfile(p)))

    if not matches:
        raise FileNotFoundError(
            f"{label}를 찾지 못했습니다.\n"
            f"기본 경로: {preferred}\n"
            "Google Drive에 파일이 존재하는지 확인하세요."
        )

    # 가장 최근 수정된 파일 우선
    matches.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    print(f"✅ {label} 자동 발견:", matches[0])
    if len(matches) > 1:
        print(f"   후보 {len(matches)}개 중 최근 파일 선택")
    return matches[0]

RQ1_PKL_PATH = find_one(
    DEFAULT_PKL,
    [
        "/content/drive/MyDrive/**/rq1_multimodel_results_3seeds.pkl",
        "/content/drive/MyDrive/**/rq1*multimodel*3seeds*.pkl",
    ],
    "RQ1 PKL"
)

DATA_PATH = find_one(
    DEFAULT_DATA,
    [
        "/content/drive/MyDrive/**/ETH MixingTornadoCash DF.csv",
        "/content/drive/MyDrive/**/*TornadoCash*.csv",
    ],
    "원본 CSV"
)

# PKL이 있는 폴더를 RQ2 결과의 기준 OUTPUT_DIR로 사용
OUTPUT_DIR = os.path.dirname(RQ1_PKL_PATH)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "_checkpoints")
RQ2_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "RQ2_Evidence_Groundedness")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RQ2_OUTPUT_DIR, exist_ok=True)

print("\n================ AUTO CONFIG ================")
print("DATA_PATH    :", DATA_PATH)
print("RQ1_PKL_PATH :", RQ1_PKL_PATH)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("RQ2_OUT      :", RQ2_OUTPUT_DIR)
print("MODE         :", "PILOT" if RQ2_PILOT else "FULL")
print("=============================================")


In [ ]:
def atomic_pickle_dump(obj, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=os.path.dirname(path))
    os.close(fd)
    try:
        with open(tmp, "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def atomic_json_dump(obj, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=os.path.dirname(path))
    os.close(fd)
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def atomic_csv_save(df, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".csv", dir=os.path.dirname(path))
    os.close(fd)
    try:
        df.to_csv(tmp, index=False, encoding="utf-8-sig")
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def load_csv_or_empty(path, columns=None):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ checkpoint loaded: {os.path.basename(path)} ({len(df):,} rows)")
        return df
    return pd.DataFrame(columns=columns)

def keyset(df, cols):
    if df is None or len(df) == 0 or any(c not in df.columns for c in cols):
        return set()
    return set(map(tuple, df[cols].astype(str).itertuples(index=False, name=None)))

## 3. 원본 데이터와 RQ1 PKL 로드
RQ1은 재학습하지 않습니다. PKL의 E4 성능으로 detector를 자동 선택하고, 그 detector의 E2/seed42 run을 RQ2에서 사용합니다.

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"DATA_PATH not found: {DATA_PATH}")
if not os.path.exists(RQ1_PKL_PATH):
    raise FileNotFoundError(f"RQ1 PKL not found: {RQ1_PKL_PATH}")

df_raw = pd.read_csv(DATA_PATH)
with open(RQ1_PKL_PATH, "rb") as f:
    results = pickle.load(f)

print("RQ1 runs loaded:", len(results))
print("First run keys:", results[0].keys())

In [ ]:
DATE_COL = "Date"
LABEL_COL = "label"
ADDRESS_COLS = ["Receiving Address", "Counterparty Address"]
ALWAYS_DROP_COLS = [
    "Hash", "Source", "Counterparty Cluster Name",
    "Counterparty Shared Name", "Counterparty Category",
]

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

def add_time_features(frame):
    out = frame.copy()
    dt = pd.to_datetime(out[DATE_COL], errors="coerce")
    out["Year"] = dt.dt.year
    out["Month"] = dt.dt.month
    out["Day"] = dt.dt.day
    out["Hour"] = dt.dt.hour
    out["DayOfWeek"] = dt.dt.dayofweek
    return out

def build_feature_frames(train_df, test_df, include_address):
    tr, te = add_time_features(train_df), add_time_features(test_df)
    y_train, y_test = tr[LABEL_COL].astype(int).copy(), te[LABEL_COL].astype(int).copy()
    drop_cols = [LABEL_COL, DATE_COL] + ALWAYS_DROP_COLS
    if not include_address:
        drop_cols += ADDRESS_COLS
    X_train = tr.drop(columns=drop_cols, errors="ignore").copy()
    X_test = te.drop(columns=drop_cols, errors="ignore").copy()
    if include_address:
        addr_cols = [x for x in ADDRESS_COLS if x in X_train.columns]
        if addr_cols:
            X_train[addr_cols] = X_train[addr_cols].fillna("__MISSING__").astype(str)
            X_test[addr_cols] = X_test[addr_cols].fillna("__MISSING__").astype(str)
            enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            X_train[addr_cols] = enc.fit_transform(X_train[addr_cols])
            X_test[addr_cols] = enc.transform(X_test[addr_cols])
    obj_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    if obj_cols:
        X_train = X_train.drop(columns=obj_cols, errors="ignore")
        X_test = X_test.drop(columns=obj_cols, errors="ignore")
    X_test = X_test.reindex(columns=X_train.columns)
    return X_train, X_test, y_train, y_test

def split_random(data, split_seed):
    return tuple(x.copy() for x in train_test_split(
        data, test_size=0.30, random_state=split_seed, stratify=data[LABEL_COL]
    ))

def model_input_from_raw(run, X_df):
    arr = run["imputer"].transform(X_df)
    if run["scaler"] is not None:
        arr = run["scaler"].transform(arr)
    return arr

def get_result_run(results, model_name, experiment_name, seed):
    found = [r for r in results if r["model_name"] == model_name and r["name"] == experiment_name and r["metrics"]["seed"] == seed]
    if len(found) != 1:
        raise ValueError(f"Expected one run, found {len(found)} for {model_name}/{experiment_name}/seed={seed}")
    return found[0]

In [ ]:
# 원본 노트북의 AUTO detector 선택 로직을 PKL에서 직접 재현
e4_name = "E4_Temporal_AddressDisjoint_AddressRemoved"
e4_rows = []
for r in results:
    if r["name"] == e4_name:
        m = r["metrics"]
        e4_rows.append({
            "model": r["model_name"], "seed": m["seed"],
            "pr_auc": m.get("pr_auc", np.nan),
            "anomaly_recall": m.get("anomaly_recall", np.nan),
            "anomaly_f1": m.get("anomaly_f1", np.nan),
            "macro_f1": m.get("macro_f1", np.nan),
        })

e4_df = pd.DataFrame(e4_rows)
if e4_df.empty:
    raise ValueError("E4 results not found in RQ1 PKL")

e4_mean = e4_df.groupby("model", as_index=False)[["pr_auc","anomaly_recall","anomaly_f1","macro_f1"]].mean()
e4_rank = e4_mean.sort_values(["pr_auc","anomaly_recall","anomaly_f1","macro_f1"], ascending=False).reset_index(drop=True)
SELECTED_MODEL = e4_rank.iloc[0]["model"]

display(e4_rank.round(4))
print("SELECTED_MODEL:", SELECTED_MODEL)

rq2_run = get_result_run(results, SELECTED_MODEL, RQ2_EXPERIMENT, RQ2_MODEL_SEED)
print("RQ2 run:", rq2_run["model_name"], rq2_run["name"], rq2_run["metrics"]["seed"])

## 4. E2 split 재구성 + SHAP
PKL에 저장된 `imputer`, `scaler`, `model`, `feature_names`, `y_pred`, `y_prob`를 그대로 사용합니다.

In [ ]:
rq2_train_df, rq2_test_df = split_random(df, RQ2_SPLIT_SEED)
X_train_rq2, X_test_rq2, y_train_rq2, y_test_rq2 = build_feature_frames(rq2_train_df, rq2_test_df, include_address=False)

# PKL feature order와 현재 재구성 feature order가 같은지 강제 확인
if X_train_rq2.columns.tolist() != rq2_run["feature_names"]:
    raise ValueError("Feature columns/order differ from RQ1 PKL. Check preprocessing/data version.")

# split/label 정합성 확인
y_saved = np.asarray(rq2_run["y_test"]).astype(int)
y_rebuilt = y_test_rq2.reset_index(drop=True).to_numpy().astype(int)
if len(y_saved) != len(y_rebuilt) or not np.array_equal(y_saved, y_rebuilt):
    raise ValueError("Rebuilt E2 y_test does not match RQ1 PKL. Do not continue; check DATA_PATH and split seed.")

X_train_arr_rq2 = model_input_from_raw(rq2_run, X_train_rq2)
X_test_arr_rq2 = model_input_from_raw(rq2_run, X_test_rq2)
feature_names = rq2_run["feature_names"]

rng = np.random.default_rng(RQ2_MODEL_SEED)
bg_idx = rng.choice(len(X_train_arr_rq2), size=min(SHAP_BACKGROUND_N, len(X_train_arr_rq2)), replace=False)
global_idx = rng.choice(len(X_test_arr_rq2), size=min(SHAP_GLOBAL_N, len(X_test_arr_rq2)), replace=False)
X_bg, X_global = X_train_arr_rq2[bg_idx], X_test_arr_rq2[global_idx]

def positive_prob_fn(X):
    return rq2_run["model"].predict_proba(np.asarray(X))[:, 1]

tree_models = ["RandomForest", "XGBoost", "LightGBM"]
if SELECTED_MODEL in tree_models:
    shap_explainer = shap.TreeExplainer(rq2_run["model"])
    raw_exp = shap_explainer(X_global)
    if raw_exp.values.ndim == 3:
        shap_exp = shap.Explanation(values=raw_exp.values[:, :, 1], base_values=np.asarray(raw_exp.base_values)[:, 1], data=X_global, feature_names=feature_names)
    else:
        shap_exp = shap.Explanation(values=raw_exp.values, base_values=raw_exp.base_values, data=X_global, feature_names=feature_names)
else:
    masker = shap.maskers.Independent(X_bg)
    shap_explainer = shap.Explainer(positive_prob_fn, masker=masker, feature_names=feature_names, algorithm="permutation")
    shap_exp = shap_explainer(X_global, max_evals=max(2 * len(feature_names) + 1, 100))

print("SHAP shape:", shap_exp.values.shape)

## 5. 100개 Evidence Package 생성

In [ ]:
y_test_np = y_test_rq2.reset_index(drop=True).to_numpy()
rng = np.random.default_rng(RQ2_MODEL_SEED)
normal_idx = np.where(y_test_np == 0)[0]
anomaly_idx = np.where(y_test_np == 1)[0]

if len(normal_idx) < N_NORMAL or len(anomaly_idx) < N_ANOMALY:
    print(f"WARNING: available normal={len(normal_idx)}, anomaly={len(anomaly_idx)}")

sample_normal = rng.choice(normal_idx, size=min(N_NORMAL, len(normal_idx)), replace=False)
sample_anomaly = rng.choice(anomaly_idx, size=min(N_ANOMALY, len(anomaly_idx)), replace=False)
sample_idx = np.concatenate([sample_normal, sample_anomaly])
rng.shuffle(sample_idx)
X_sample = X_test_arr_rq2[sample_idx]

if SELECTED_MODEL in tree_models:
    raw_sample = shap_explainer(X_sample)
    sample_shap = raw_sample.values[:, :, 1] if raw_sample.values.ndim == 3 else raw_sample.values
else:
    sample_exp = shap_explainer(X_sample, max_evals=max(2 * len(feature_names) + 1, 100))
    sample_shap = sample_exp.values

def safe_scalar(v):
    if pd.isna(v): return None
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return float(v)
    return v

X_test_raw_reset = X_test_rq2.reset_index(drop=True)

def build_evidence_package(local_pos, global_idx, top_k=5):
    row = X_test_raw_reset.iloc[global_idx]
    sv = sample_shap[local_pos]
    order = np.argsort(np.abs(sv))[::-1][:top_k]
    top_evidence = []
    for rank, j in enumerate(order, 1):
        top_evidence.append({
            "rank": rank, "feature": feature_names[j],
            "value": safe_scalar(row.iloc[j]), "shap_value": float(sv[j]),
            "effect": "toward_anomaly" if sv[j] > 0 else "toward_normal",
        })
    return {
        "case_id": int(global_idx), "detector": SELECTED_MODEL,
        "true_label": "Anomaly" if int(y_test_np[global_idx]) == 1 else "Normal",
        "prediction": "Anomaly" if int(rq2_run["y_pred"][global_idx]) == 1 else "Normal",
        "anomaly_probability": float(rq2_run["y_prob"][global_idx]),
        "top_evidence": top_evidence,
    }

evidence_packages = [build_evidence_package(i, int(idx), TOP_K_EVIDENCE) for i, idx in enumerate(sample_idx)]
EVIDENCE_CKPT = os.path.join(CHECKPOINT_DIR, "evidence_packages.pkl")
atomic_pickle_dump(evidence_packages, EVIDENCE_CKPT)
print("Evidence packages:", len(evidence_packages))
print(json.dumps(evidence_packages[0], ensure_ascii=False, indent=2))
print("Saved:", EVIDENCE_CKPT)

## 6. OpenRouter — 자동 Key 로딩
환경변수 또는 Colab Secret의 `OPENROUTER_API_KEY`를 자동으로 사용합니다. 없으면 실행 중 한 번 입력창이 뜹니다.


In [ ]:

import os, getpass

def load_openrouter_key():
    # 1) Environment variable
    key = os.getenv("OPENROUTER_API_KEY", "").strip()
    if key:
        print("✅ OPENROUTER_API_KEY: environment variable")
        return key

    # 2) Colab Secret
    try:
        from google.colab import userdata
        key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
        if key:
            print("✅ OPENROUTER_API_KEY: Colab Secret")
            return key
    except Exception:
        pass

    # 3) No source edit required: secure prompt
    key = getpass.getpass("OpenRouter API Key 입력 (화면에 표시되지 않음): ").strip()
    if not key:
        raise ValueError("OpenRouter API Key가 필요합니다.")
    return key

OPENROUTER_API_KEY = load_openrouter_key()

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
QWEN_MODEL = "qwen/qwen3-235b-a22b-2507"
LLAMA_JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct"
GEN_TEMPERATURE = 0.2
JUDGE_TEMPERATURE = 0.0

openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL
)

def qwen_generate(prompt, temperature=GEN_TEMPERATURE, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = openrouter_client.chat.completions.create(
                model=QWEN_MODEL,
                messages=[{"role":"user","content":prompt}],
                temperature=temperature
            )
            text = r.choices[0].message.content
            if text is None:
                raise RuntimeError("Qwen returned empty output")
            return text.strip()
        except Exception as e:
            print(f"Qwen/OpenRouter error ({attempt+1}/{max_retries}): {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(min(60, 5 * (2 ** attempt)))

def llama_judge(prompt, temperature=JUDGE_TEMPERATURE, max_retries=6):
    for attempt in range(max_retries):
        try:
            r = openrouter_client.chat.completions.create(
                model=LLAMA_JUDGE_MODEL,
                messages=[{"role":"user","content":prompt}],
                temperature=temperature
            )
            text = r.choices[0].message.content
            if text is None:
                raise RuntimeError("Llama returned empty output")
            return text.strip()
        except Exception as e:
            print(f"Llama/OpenRouter error ({attempt+1}/{max_retries}): {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(min(60, 5 * (2 ** attempt)))

# API 연결 사전 확인
print("Qwen Generator:", QWEN_MODEL)
print("Llama Judge   :", LLAMA_JUDGE_MODEL)
print("Testing OpenRouter connection...")
print("Qwen :", qwen_generate("Reply with exactly: QWEN OK", temperature=0.0))
print("Llama:", llama_judge("Reply with exactly: LLAMA OK", temperature=0.0))


## 7. Baseline / HCLA Narrative 생성

In [ ]:
HCLA_SYSTEM = """
You are the Justification component of HCLA for transaction anomaly analysis.
Generate a concise explanation using ONLY the supplied evidence.
Do not introduce external facts, address history, criminal attribution,
causal claims, or domain facts not explicitly present in the evidence package.
If evidence is insufficient, state that limitation.
Explain the evidence in plain language.
""".strip()

BASELINE_SYSTEM = """
You are an AI assistant explaining a transaction anomaly prediction.
Use the supplied transaction information to provide a concise and useful explanation
for why the model may have produced this prediction.
""".strip()

def make_narrative_prompt(evidence, condition):
    system = HCLA_SYSTEM if condition == "HCLA" else BASELINE_SYSTEM
    return f"""{system}

Evidence package:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

Return one short paragraph (2-4 sentences).
"""

def generate_narratives(evidence_list, n_generations=3, checkpoint_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq2_narratives.csv")
    cols = ["case_id","true_label","prediction","condition","rep","narrative","evidence_json"]
    df0 = load_csv_or_empty(checkpoint_path, cols)
    rows = df0.to_dict("records") if len(df0) else []
    done = keyset(df0, ["case_id","condition","rep"])
    total = len(evidence_list) * 2 * n_generations
    print(f"RQ2 Narrative resume {len(done)}/{total}")
    for e in evidence_list:
        for condition in ["Baseline", "HCLA"]:
            for rep in range(1, n_generations + 1):
                key = (str(e["case_id"]), condition, str(rep))
                if key in done: continue
                text = qwen_generate(make_narrative_prompt(e, condition))
                rows.append({
                    "case_id":e["case_id"], "true_label":e["true_label"], "prediction":e["prediction"],
                    "condition":condition, "rep":rep, "narrative":text,
                    "evidence_json":json.dumps(e, ensure_ascii=False),
                })
                done.add(key)
                atomic_csv_save(pd.DataFrame(rows, columns=cols), checkpoint_path)
                print(f"✅ RQ2 Narrative {len(done)}/{total}")
    return pd.DataFrame(rows, columns=cols)

## 8. Claim-level Grounding Judge

In [ ]:
def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m: raise ValueError(text[:1000])
        return json.loads(m.group(0))

GROUNDING_JUDGE = """
You are a strict evidence-grounding evaluator.
Evaluate the explanation ONLY against the supplied evidence package.

Split the explanation into atomic factual claims.
For each claim, label it:
- SUPPORTED
- UNSUPPORTED
- CONTRADICTED

Also list which supplied top-evidence feature names were actually used.

Return ONLY JSON:
{
  "claims": [
    {"claim":"...", "label":"SUPPORTED|UNSUPPORTED|CONTRADICTED", "reason":"..."}
  ],
  "used_evidence_features": ["..."]
}
""".strip()

def judge_narrative(evidence, narrative):
    prompt = f"""{GROUNDING_JUDGE}

Evidence package:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

Explanation:
{narrative}
"""
    return extract_json_object(llama_judge(prompt, temperature=0.0))

def evaluate_narratives(narratives_df, checkpoint_path=None, progress_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq2_claims.csv")
    progress_path = progress_path or os.path.join(CHECKPOINT_DIR, "rq2_judge_done.json")
    cols = ["case_id","condition","rep","claim","label","reason","used_evidence_features","evidence_json","narrative"]
    df0 = load_csv_or_empty(checkpoint_path, cols)
    rows = df0.to_dict("records") if len(df0) else []
    if os.path.exists(progress_path):
        with open(progress_path, "r", encoding="utf-8") as f:
            done = {tuple(map(str, x)) for x in json.load(f)}
    else:
        done = set()
    total = len(narratives_df)
    print(f"RQ2 Judge resume {len(done)}/{total}")
    for _, row in narratives_df.iterrows():
        key = (str(row["case_id"]), str(row["condition"]), str(row["rep"]))
        if key in done: continue
        evidence = json.loads(row["evidence_json"])
        judged = judge_narrative(evidence, row["narrative"])
        used = judged.get("used_evidence_features", [])
        claims = judged.get("claims", []) or [{"claim":row["narrative"],"label":"UNSUPPORTED","reason":"No claims parsed"}]
        for x in claims:
            rows.append({
                "case_id":row["case_id"], "condition":row["condition"], "rep":row["rep"],
                "claim":x.get("claim", ""), "label":str(x.get("label", "")).upper(), "reason":x.get("reason", ""),
                "used_evidence_features":json.dumps(used, ensure_ascii=False),
                "evidence_json":row["evidence_json"], "narrative":row["narrative"],
            })
        done.add(key)
        atomic_csv_save(pd.DataFrame(rows, columns=cols), checkpoint_path)
        atomic_json_dump(sorted([list(x) for x in done]), progress_path)
        print(f"✅ RQ2 Judge {len(done)}/{total}")
    return pd.DataFrame(rows, columns=cols)

def summarize_groundedness(claims_df):
    rows = []
    for key, g in claims_df.groupby(["case_id","condition","rep"]):
        labels = g["label"].tolist(); n = len(labels)
        evidence = json.loads(g.iloc[0]["evidence_json"])
        supplied = {x["feature"] for x in evidence["top_evidence"]}
        used = set()
        for s in g["used_evidence_features"]:
            try: used.update(json.loads(s))
            except Exception: pass
        rows.append({
            "case_id":key[0], "condition":key[1], "rep":key[2],
            "ESR":labels.count("SUPPORTED")/n if n else np.nan,
            "UCR":labels.count("UNSUPPORTED")/n if n else np.nan,
            "CR":labels.count("CONTRADICTED")/n if n else np.nan,
            "EC":len(supplied.intersection(used))/len(supplied) if supplied else np.nan,
        })
    metric_df = pd.DataFrame(rows)
    mean_df = metric_df.groupby("condition")[["ESR","UCR","CR","EC"]].mean()
    sd_df = metric_df.groupby("condition")[["ESR","UCR","CR","EC"]].std(ddof=1)
    paper = pd.DataFrame(index=mean_df.index)
    for metric in ["ESR","UCR","CR","EC"]:
        paper[metric] = [f"{mean_df.loc[x,metric]:.4f} ± {sd_df.loc[x,metric]:.4f}" for x in mean_df.index]
    return metric_df, mean_df, sd_df, paper.reset_index()

## 9. RQ2 실행
Checkpoint가 있으면 완료된 `case × condition × rep`은 자동으로 건너뜁니다.

In [ ]:
rq2_evidence = evidence_packages[:10] if RQ2_PILOT else evidence_packages[:100]
rq2_generations = 1 if RQ2_PILOT else N_GENERATIONS

RQ2_NARRATIVE_CKPT = os.path.join(CHECKPOINT_DIR, "rq2_narratives.csv")
RQ2_CLAIMS_CKPT = os.path.join(CHECKPOINT_DIR, "rq2_claims.csv")
RQ2_JUDGE_DONE = os.path.join(CHECKPOINT_DIR, "rq2_judge_done.json")

print("Cases:", len(rq2_evidence))
print("Generations per condition:", rq2_generations)
print("Expected explanations:", len(rq2_evidence) * 2 * rq2_generations)

narratives_df = generate_narratives(rq2_evidence, n_generations=rq2_generations, checkpoint_path=RQ2_NARRATIVE_CKPT)
claims_df = evaluate_narratives(narratives_df, checkpoint_path=RQ2_CLAIMS_CKPT, progress_path=RQ2_JUDGE_DONE)
metric_df, rq2_mean_df, rq2_sd_df, rq2_paper_df = summarize_groundedness(claims_df)

atomic_csv_save(narratives_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_narratives_full.csv"))
atomic_csv_save(claims_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_claims_full.csv"))
atomic_csv_save(metric_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_metrics_by_case.csv"))
atomic_csv_save(rq2_paper_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_paper_summary.csv"))

print("✅ RQ2 COMPLETE")
display(rq2_paper_df)

In [ ]:

print("\n" + "="*70)
print("RQ2 FINAL INTERPRETATION")
print("="*70)
print("ESR: Evidence Support Rate      → 높을수록 좋음")
print("UCR: Unsupported Claim Rate    → 낮을수록 좋음")
print("CR : Contradiction Rate        → 낮을수록 좋음")
print("EC : Evidence Coverage         → 높을수록 좋음")
print("\nBaseline과 HCLA를 비교해 HCLA가")
print("  ① ESR↑  ② UCR↓  ③ CR↓  ④ EC↑")
print("이면 evidence-grounded explanation 효과를 뒷받침합니다.")
print("\n결과 파일:")
print(" -", os.path.join(RQ2_OUTPUT_DIR, "rq2_paper_summary.csv"))
print(" -", os.path.join(RQ2_OUTPUT_DIR, "rq2_metrics_by_case.csv"))
print(" -", os.path.join(RQ2_OUTPUT_DIR, "rq2_claims_full.csv"))
print("="*70)


## 10. 선택 사항 — Human validation sample
Llama judge 검증용 표본을 조건별 최대 30 claim씩 저장합니다. `human_label`을 사람이 입력한 뒤 다음 셀에서 Cohen's κ를 계산할 수 있습니다.

In [ ]:
HUMAN_SAMPLE_PER_CONDITION = 30
human_parts = []
for condition, g in claims_df.groupby("condition"):
    human_parts.append(g.sample(n=min(HUMAN_SAMPLE_PER_CONDITION, len(g)), random_state=RQ2_MODEL_SEED))

human_review_df = pd.concat(human_parts, ignore_index=True).copy()
human_review_df["human_label"] = ""
human_review_df["reviewer_note"] = ""
human_path = os.path.join(RQ2_OUTPUT_DIR, "rq2_llama_human_validation_sample.csv")
human_review_df.to_csv(human_path, index=False, encoding="utf-8-sig")
print("Human validation sample saved:", human_path)

In [ ]:
def evaluate_human_agreement(reviewed_df):
    x = reviewed_df.copy()
    x["human_label"] = x["human_label"].astype(str).str.upper().str.strip()
    x = x[x["human_label"].isin(["SUPPORTED","UNSUPPORTED","CONTRADICTED"])].copy()
    if len(x) == 0:
        print("No completed human labels.")
        return None
    result = pd.DataFrame([{
        "judge_model":LLAMA_JUDGE_MODEL, "n_reviewed":len(x),
        "raw_agreement":(x["label"] == x["human_label"]).mean(),
        "cohen_kappa":cohen_kappa_score(x["human_label"], x["label"]),
    }])
    display(result)
    return result

# 사람이 human_label을 입력한 CSV를 다시 읽은 뒤 실행:
# reviewed = pd.read_csv(human_path)
# evaluate_human_agreement(reviewed)

## 산출물
- `rq2_narratives_full.csv` — Qwen이 생성한 Baseline/HCLA 설명
- `rq2_claims_full.csv` — claim별 SUPPORTED / UNSUPPORTED / CONTRADICTED 판정
- `rq2_metrics_by_case.csv` — case별 ESR/UCR/CR/EC
- `rq2_paper_summary.csv` — 논문 표에 바로 쓸 평균±표준편차 요약
- `rq2_llama_human_validation_sample.csv` — 선택적 Human validation 표본

Checkpoint가 있으므로 Colab 연결이 끊겨도 **모두 실행을 다시 누르면 완료된 항목을 건너뛰고 이어서 실행**합니다.
